# 🏰 천상계 5차시 · 바람의 도시 — 현의 길이와 스트링아트

**원, 더 월드 · 천상계(Manim 영상 제작 실습)** | 이름: ____________________ `v1 (2026-09-21)`

🗺️ **연결된 도시**: CITY 05 바람의 도시 — 원과 직선 — 현의 길이 (게임: 꿀꿀쥐 방어전)

📐 **오늘의 수학**: 한 원에서 **길이가 같은 두 현은 중심에서 같은 거리**에 있고, 중심에 가까운 현일수록 길다.

🎯 **학습 목표**

- 길이가 같은 두 현의 중심거리가 같음을 회전 애니메이션으로 보일 수 있다.
- 현을 움직이며 중심거리와 현의 길이의 관계를 숫자판으로 확인할 수 있다.
- 원 위의 점을 규칙적으로 이어 스트링아트 영상을 만들 수 있다.

> 🌌 바람의 도시에서 꿀꿀쥐를 정화하며 그은 현들 — 그 길이의 비밀과, 현이 모여 만드는 스트링아트를 영상으로 만들어요.

In [ ]:
#@title ⚙️ 1단계: Manim 설치 (재생 ▶ 누르고 약 2분 대기)
# ⚠️ 이 셀은 고치지 말고 그대로 실행하세요. 빨간 경고가 나와도 괜찮아요.
!sudo apt update -qq
!sudo apt install -y -qq libcairo2-dev libpango1.0-dev ffmpeg fonts-nanum > /dev/null
!fc-cache -f > /dev/null
!pip install -q manim
print("📦 설치 끝!")
print("👉 이제 메뉴 [런타임] → [세션 다시 시작] 을 누른 뒤,")
print("   아래 '2단계' 셀부터 실행하세요. (이 셀은 다시 실행하지 않아도 돼요)")

> ### 🔄 잠깐! 설치가 끝나면 꼭 **[런타임] → [세션 다시 시작]** 을 누르세요.
> 새로 설치된 도구를 Colab이 제대로 불러오려면 한 번 재시작해야 해요.
> 재시작 후에는 위의 설치 셀 말고 **아래 셀부터** 실행하면 됩니다.

In [ ]:
#@title ✅ 2단계: 준비 확인 (세션 다시 시작 후 실행)
from manim import *
import numpy as np
print("✅ 준비 완료! 이제 아래 셀들을 실행할 수 있어요.")

## 1. 길이가 같은 두 현은 중심에서 같은 거리

두 현 AB, CD의 길이가 같으면, 중심 O에서 두 현까지의 거리(OM, ON)도 같아요.
(△OAM ≡ △OCN: 빗변 OA = OC(반지름), AM = CN(현의 절반))

한 현을 **중심 둘레로 회전**시키면 다른 현과 정확히 겹쳐요 — 그래서 거리도 같아요!

In [ ]:
%%manim -qm -v WARNING EqualChords

class EqualChords(Scene):
    def construct(self):
        r = 2.6
        circ = Circle(radius=r, color=GREY)
        O = ORIGIN
        self.play(Create(circ), FadeIn(Dot(O, color=ORANGE)))

        def chord_at(mid_deg, half_deg, color):
            a, b = (mid_deg - half_deg) * DEGREES, (mid_deg + half_deg) * DEGREES
            A = r * np.array([np.cos(a), np.sin(a), 0]); B = r * np.array([np.cos(b), np.sin(b), 0])
            M = (A + B) / 2
            return VGroup(Line(A, B, color=color, stroke_width=6), Line(O, M, color=color), Dot(M, color=color))

        g1 = chord_at(60, 40, YELLOW)     # 현 AB + 중심거리 OM
        g2 = chord_at(200, 40, GREEN)     # 현 CD + 중심거리 ON (같은 길이)
        self.play(Create(g1), Create(g2))
        d = np.linalg.norm(g1[1].get_end())
        t = Text(f"OM = ON = {d:.2f}", font_size=30).to_edge(UP)
        self.play(Write(t))
        self.wait(0.5)
        self.play(Rotate(g1.copy(), angle=140 * DEGREES, about_point=O), run_time=2.5)   # 회전해서 겹치기
        msg = Text("돌리면 딱 겹친다 → 중심거리도 같다!", font="NanumGothic", font_size=30, color=YELLOW).to_edge(DOWN)
        self.play(Write(msg))
        self.wait(2)

## 2. ⭐ 말없는 증명 — 중심에 가까울수록 현은 길다

현을 아래에서 위로 밀어 올리면(중심에 가까워지면) 현이 점점 길어지고, 중심을 지날 때 가장 길어요(= 지름).
`ValueTracker`로 현의 높이 `h`를 움직이며 **중심거리 |h|와 현의 길이**를 숫자로 확인해요.

$$\text{현의 길이} = 2\sqrt{r^2 - d^2} \quad (d: \text{중심에서 현까지의 거리})$$

In [ ]:
%%manim -qm -v WARNING SlidingChord

class SlidingChord(Scene):
    def construct(self):
        r = 2.6
        shift = LEFT * 2.5
        circ = Circle(radius=r, color=GREY).shift(shift)
        O = Dot(color=ORANGE).shift(shift)
        self.play(Create(circ), FadeIn(O))

        h = ValueTracker(-2.3)                  # 현의 높이 (중심거리 = |h|)
        def chord():
            y = h.get_value(); half = np.sqrt(r ** 2 - y ** 2)
            return Line([-half, y, 0], [half, y, 0], color=YELLOW, stroke_width=7).shift(shift)
        def dist():
            y = h.get_value()
            return Line([0, 0, 0], [0, y, 0], color=BLUE).shift(shift)
        def panel():
            y = h.get_value(); L = 2 * np.sqrt(r ** 2 - y ** 2)
            return Text(f"중심거리  {abs(y):4.2f}\n현의 길이  {L:4.2f}", font="NanumGothic",
                        font_size=32, line_spacing=1.2).move_to(RIGHT * 3.3)
        self.add(always_redraw(chord), always_redraw(dist), always_redraw(panel))
        self.play(h.animate.set_value(0), run_time=3)
        self.wait(0.5)
        self.play(h.animate.set_value(2.3), run_time=3)
        msg = Text("중심에 가까울수록 길다 · 지름이 가장 긴 현", font="NanumGothic", font_size=28, color=YELLOW).to_edge(DOWN)
        self.play(Write(msg))
        self.wait(2)

## 3. 스트링아트 — 현이 모여 곡선이 된다 🧵

원 둘레에 점 N개를 번호 매겨 찍고, 각 점 i를 **(i × k)를 N으로 나눈 나머지** 번호의 점과 현으로 이어요.

- k = 2 → 하트(심장형) 곡선, k = 3 → 콩팥형 곡선
- 곧은 현들이 모여 부드러운 곡선(포락선)이 보여요. 바람의 도시 영상이 바로 이거예요!

In [ ]:
%%manim -qm -v WARNING TimesTableCircle

class TimesTableCircle(Scene):
    def construct(self):
        N, k, R = 100, 2, 3.0            # 점 개수, 곱하는 수, 반지름

        def point(i):
            ang = PI / 2 - TAU * i / N
            return R * np.array([np.cos(ang), np.sin(ang), 0])

        circle = Circle(radius=R, color=GREY)
        label = Text(f"× {k}  (점 {N}개)", font="NanumGothic", font_size=30).to_edge(UP)
        self.play(Create(circle), Write(label))

        strings = VGroup(*[Line(point(i), point((i * k) % N), stroke_width=1.2) for i in range(N)])
        strings.set_color_by_gradient(TEAL, BLUE, PURPLE)
        self.play(Create(strings), run_time=5)
        self.wait(1)

## 🏆 도전 과제 — 나만의 스트링아트

1. `N`, `k`, 색을 바꿔 무늬를 만들어 보세요. 추천: `k=33, N=100` / `k=51, N=200` / `k=99, N=200`
2. `k`를 `ValueTracker`로 2 → 40까지 움직이면 무늬가 살아 움직여요. (아래 `StringArtShow` 참고)
3. (보너스) 완성작을 mp4로 저장해 은하계 영상으로 제출해 보세요!

In [ ]:
%%manim -qm -v WARNING StringArtShow

class StringArtShow(Scene):
    def construct(self):
        N, R = 120, 3.0
        k = ValueTracker(2)
        my_colors = [YELLOW, ORANGE, RED, PINK]      # 색을 바꿔 보세요

        def point(i):
            ang = PI / 2 - TAU * i / N
            return R * np.array([np.cos(ang), np.sin(ang), 0])
        def make():
            kk = k.get_value()
            g = VGroup(*[Line(point(i), point((i * kk) % N), stroke_width=1.0) for i in range(N)])
            return g.set_color_by_gradient(*my_colors)
        label = always_redraw(lambda: Text(f"× {k.get_value():.1f}", font="NanumGothic", font_size=30).to_edge(UP))
        self.add(Circle(radius=R, color=GREY), always_redraw(make), label)
        self.play(k.animate.set_value(40), run_time=12, rate_func=linear)   # 끝값을 바꿔 보세요
        self.wait(1)

## 📝 오늘 배운 것 정리

- 길이가 같은 두 현은 중심에서 같은 거리에 있다 (회전하면 겹친다).
- 중심에 가까운 현일수록 길고, **지름이 가장 긴 현**이다: 현의 길이 = 2√(r² − d²).
- 원 위 점을 `(i × k) mod N` 규칙으로 이으면 현들이 곡선(포락선)을 만든다.
- `Rotate`, `ValueTracker`, `set_color_by_gradient`, `rate_func=linear`를 썼다.

**다음 시간 (비눗방울 도시)**: 원 밖의 한 점에서 그은 **두 접선의 길이**가 같음을 영상으로!